In [1]:
# 1. Uninstall the conflicting versions
!pip uninstall -y spacy thinc numpy
# 2. Install latest versions (letting pip resolve the best match)
!pip install -U numpy spacy
# 3. Download the model
!python -m spacy download en_core_web_sm

Found existing installation: spacy 3.8.11
Uninstalling spacy-3.8.11:
  Successfully uninstalled spacy-3.8.11
Found existing installation: thinc 8.3.10
Uninstalling thinc-8.3.10:
  Successfully uninstalled thinc-8.3.10
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 65.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


^C


In [1]:
import spacy
import nltk
import zipfile
import os
import json
import re
from pathlib import Path
import pandas as pd
from spacy.matcher import Matcher
from nltk import pos_tag
from tqdm import tqdm
from google.colab import files # For downloading outputs later

In [2]:
# Load models
print(spacy.__version__)
nlp = spacy.load("en_core_web_sm")
nltk.download('averaged_perceptron_tagger_eng')

3.8.11


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

# ZIP EXTRACTION

In [3]:
# ZIP EXTRACTION
zip_file_path = '/content/txt_output_cleaned.zip'
extract_dir = '/content/txt_output_cleaned'

files_to_process = [
    '1102.1027v1_Collective Classification of Textual Documents by .tei.txt',
    '1103.4090v2_A Linear Classifier Based on Entity Recognition To.tei.txt',
    '1110.2626v1_Analysis of Heart Diseases Dataset using Neural Ne.tei.txt',
    '1110.3088v1_Towards cross-lingual alerting for bursty epidemic.tei.txt',
    '1110.3569v1_Dimension Reduction of Health Data Clustering.tei.txt',
    '1112.4261v1_Performance Analysis of Enhanced Clustering Algori.tei.txt',
    '1203.6845v1_Information Retrieval Systems Adapted to the Biome.tei.txt',
    '1207.3285v1_Biogeography-Based Informative Gene Selection and .tei.txt',
    '1302.6426v1_Segmentation of Alzheimers Disease in PET scan dat.tei.txt',
    '1302.7082v1_K Means Segmentation of Alzheimers Disease in PET .tei.txt',
    '1304.2538v1_On Appropriate Selection of Fuzzy Aggregation Oper.tei.txt',
    '1311.7251v1_Spatially-Adaptive Reconstruction in Computed Tomo.tei.txt',
    '1402.1781v1_Discovering functional DNA elements using populati.tei.txt',
    '1402.3632v1_The Cure Making a game of gene selection for breas.tei.txt',
    '1402.5338v3_A statistical fat-tail test of predicting regulato.tei.txt',
    '1402.6151v1_Approaching allelic probabilities and Genome-Wide .tei.txt',
    '1404.0329v1_Toward computational cumulative biology by combini.tei.txt',
    '1405.5935v1_Prokaryotic regulatory systems biology Common prin.tei.txt',
    '1408.1002v1_Signalling Entropy a novel network-theoretical fra.tei.txt',
    '1501.05353v1_Rapid Sequence Identification of Potential Pathoge.tei.txt',
    '1505.06915v1_Large-scale Machine Learning for Metagenomics Sequ.tei.txt',
    '1901.03419v1.tei.txt',
    '2002.11509v1(20).tei.txt',
    '2009.10765v2_Age-Net An MRI-Based Iterative Framework for Brain.tei.txt',
    '2009.11120v2_Anisotropic 3D Multi-Stream CNN for Accurate Prost.tei.txt',
    '2010.00747v2_Contrastive Learning of Medical Visual Representat.tei.txt',
    '2010.01052v3_Joint data imputation and mechanistic modelling fo.tei.txt',
    '2010.01982v1_Automatic Deep Learning System for COVID-19 Infect.tei.txt',
    '2010.06163v2_Bridging 2D and 3D Segmentation Networks for Compu.tei.txt',
    '2010.08582v2_CT Image Segmentation for Inflamed and Fibrotic Lu.tei.txt',
    '2011.03772v2_Automated Grading System of Retinal Arterio-venous.tei.txt',
    '2012.03684v1_Multi-Decoder Networks with Multi-Denoising Inputs.tei.txt',
    '20251107_2511.05726v1_GastroDL-Fusion_ A Dual-Modal Deep Learning Framework Integrating Protein-Ligand Complexes and Gene .tei.txt',
    '20251108_2511.05960v1_Deep Survival Analysis of Longitudinal EHR Data for Joint Prediction of Hospitalization and Death in.tei.txt',
    '20251114_2511.10971v1_ERMoE_ Eigen-Reparameterized Mixture-of-Experts for Stable Routing and Interpretable Specialization.tei.txt',
    '20251114_2511.11324v1_NOVA_ An Agentic Framework for Automated Histopathology Analysis and Discovery.tei.txt',
    '2101.04702v5_Cross-Modal Contrastive Learning for Text-to-Image.tei.txt',
    '2102.08556v3_Deep cross-modality MR-CT educed distillation lear.tei.txt',
    '2102.10484v2_CheXseg Combining Expert Annotations with DNN-gene.tei.txt',
    '2102.10765v1_Post-hoc Overall Survival Time Prediction from Bra.tei.txt',
    '2102.11099v1_RCoNet Deformable Mutual Information Maximization .tei.txt',
    '2103.05232v1_Stabilized Medical Image Attacks.tei.txt',
    '2103.06575v1_An unsupervised deep learning framework for medica.tei.txt',
    '2103.10178v1_A Location-Sensitive Local Prototype Network for F.tei.txt',
    '2103.16022v1_Self-supervised Image-text Pre-training With Mixed.tei.txt',
    '2103.16617v4.tei.txt',
    '2104.10481v4_SKID Self-Supervised Learning for Knee Injury Diag.tei.txt',
    '2106.09076v3_Deformation Driven Seq2Seq Longitudinal Tumor and .tei.txt',
    '2107.02008v2_Improving a neural network model by explanation-gu.tei.txt',
    '2107.05047v1_One Map Does Not Fit All Evaluating Saliency Map E.tei.txt'
]

if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    files_in_zip = zip_ref.namelist()
    files_to_extract = [f for f in files_in_zip if f.split('/')[-1] in files_to_process]
    zip_ref.extractall(extract_dir, members=files_to_extract)

print(f"Files extracted to: {extract_dir}")
INPUT_DIR = Path('/content/txt_output_cleaned/txt_output_cleaned')

Files extracted to: /content/txt_output_cleaned


# RULE-BASED PROCESSING

In [4]:
# RULE-BASED PROCESSING
print("Starting Rule-Based Processing...")
OUTPUT_CSV_RULE = Path("RULEBASED_from_txt.csv")

Keywords = [
    r"data\s*(?:set|base)s?", r"corpus", r"corpora", r"tree\s*bank",
    r"(?:train|test|validation|testing|trainings?)\s*(?:set|data)",
    r"collections?", r"benchmarks?",
]
expression = re.compile(r"\b(" + "|".join(Keywords) + r")\b", flags=re.IGNORECASE)

pattern = [
    {"IS_PUNCT": False, "POS": {"IN": ["PROPN", "NOUN", "NUM"]}, "OP": "?"},
    {"LENGTH": {">=": 2}, "IS_LOWER": False, "IS_PUNCT": False, "IS_DIGIT": False, "POS": {"IN": ["PROPN", "NOUN", "ADJ"]}},
    {"POS": {"IN": ["PROPN", "NOUN", "NUM", "PUNCT", "ADJ"]}, "OP": "*"},
    {"TEXT": {"IN": ["and", "or"]}, "OP": "?"},
    {"POS": {"IN": ["PROPN", "NOUN", "NUM", "PUNCT", "ADJ"]}, "OP": "*"},
    {"TEXT": {"REGEX": r"((:?d|D)ata\s*(?:set|base)s?$|(:?C|c)orpus$|(:?C|c)orpora$|(:?T|t)ree\s*bank$|(:?C|c)ollections?$|(:?B|b)enchmarks?)$|trainval$"}},
]

pattern2 = [
    {"IS_PUNCT": False, "POS": {"IN": ["PROPN", "NOUN", "NUM"]}, "OP": "?"},
    {"LENGTH": {">=": 2}, "IS_LOWER": False, "IS_PUNCT": False, "IS_DIGIT": False, "POS": {"IN": ["PROPN", "NOUN", "ADJ"]}},
    {"POS": {"IN": ["PROPN", "NOUN", "NUM", "PUNCT", "ADJ"]}, "OP": "*"},
    {"TEXT": {"IN": ["and", "or"]}, "OP": "?"},
    {"POS": {"IN": ["PROPN", "NOUN", "NUM", "PUNCT", "ADJ"]}, "OP": "*"},
    {"TEXT": {"REGEX": r"(train$|test$|validation$|testing$|trainings?$|data$)"}},
    {"TEXT": {"REGEX": r"(sets?|(:?D|d)atasets?|(:?D|d)atabases?)"}},
]

rulebased_rows = []

for txt_file in tqdm(sorted(INPUT_DIR.glob("*.txt")), desc="Processing Rule-Based Files"):
    filename = txt_file.name
    text = txt_file.read_text(encoding="utf-8", errors="ignore")
    doc = nlp(text)

    sent_idx = 0 # Local counter per file
    for sent in doc.sents:
        sentence = sent.text.strip()
        if not sentence: continue

        start_index = text.find(sentence)

        sent_doc = sent.as_doc()
        tokens = [t.text for t in sent_doc]
        pos_tags = nltk.pos_tag(tokens)

        is_keyword_match = bool(re.search(expression, sentence))
        store2 = {}

        if is_keyword_match:
            matcher = Matcher(nlp.vocab)
            matcher.add("Dataset", [pattern, pattern2])
            matches = matcher(sent_doc)

            store = {}
            for _, start, end in matches:
                if end not in store or store[end] > start:
                    store[end] = start

            for end, start in store.items():
                if start not in store2 or store2[start] < end:
                    store2[start] = end

        end_idx = 0
        for i in range(len(tokens)):
            if is_keyword_match:
                if len(store2) == 0:
                    tag = "O"
                elif i in store2:
                    tag = "B"
                    end_idx = store2[i]
                elif i < end_idx:
                    tag = "I"
                else:
                    tag = "O"
            else:
                tag = "O"

            rulebased_rows.append({
                "File": filename,
                "Sentence #": sent_idx,
                "Start Index": start_index, # Now an integer
                "Word": pos_tags[i][0],
                "POS": pos_tags[i][1],
                "Tag": tag,
                "Sentence Text": sentence,
            })

        sent_idx += 1 # Increment local sentence counter

df_rule = pd.DataFrame(rulebased_rows)
output_columns = ["File", "Sentence #", "Start Index", "Word", "POS", "Tag", "Sentence Text"]
df_rule.to_csv(OUTPUT_CSV_RULE, columns=output_columns, index=False)
print(f"Saved Rule-based CSV to {OUTPUT_CSV_RULE}")

Starting Rule-Based Processing...


Processing Rule-Based Files: 100%|██████████| 50/50 [00:40<00:00,  1.25it/s]


Saved Rule-based CSV to RULEBASED_from_txt.csv


# GOLDEN PROCESSING

In [6]:
# GOLDEN PROCESSING (OVERLAP)
print("\nStarting Golden Processing...")
OUTPUT_CSV_GOLDEN = Path("Golden.csv")
golden_labels_file = '/content/annotations_backup (1).json'

with open(golden_labels_file, 'r') as f:
    golden_labels = json.load(f)

golden_rows = []
output_columns = ["File", "Sentence #", "Start Index", "Word", "POS", "Tag", "Sentence Text"]

for file, labels_data in tqdm(golden_labels.items(), desc="Processing Golden Files"):
    file_path = INPUT_DIR / file
    if not file_path.exists(): continue

    text = file_path.read_text(encoding="utf-8", errors="ignore")
    doc = nlp(text)

    # 1. Map labels to document-level tokens using an OVERLAP strategy
    token_tags = ['O'] * len(doc)

    for result in labels_data[0]['result']:
        label = result['value']
        start = label['start']
        end = label['end']

        b_assigned = False
        for i, token in enumerate(doc):
            token_start = token.idx
            token_end = token_start + len(token.text)

            # OVERLAP CONDITION: If the token touches the human annotation at all
            if token_start < end and token_end > start:
                if token.text.strip() == "":
                    continue # Skip empty space tokens

                if not b_assigned:
                    token_tags[i] = 'B'
                    b_assigned = True
                else:
                    if token_tags[i] == 'O': # Don't overwrite an existing 'B' with an 'I'
                        token_tags[i] = 'I'

    # 2. Build the CSV preserving exact sentence alignment
    sent_idx = 0
    token_global_idx = 0 # Tracks our absolute position in the document

    for sent in doc.sents:
        sentence = sent.text.strip()

        # If the sentence is just whitespace, skip adding to CSV,
        # but we MUST advance the global token counter so tags don't misalign!
        if not sentence:
            token_global_idx += len(sent)
            continue

        start_index = text.find(sentence) # Kept identical to Rule-based script

        tokens = [t.text for t in sent]
        pos_tags = pos_tag(tokens)

        for i, token in enumerate(sent):
            golden_rows.append({
                "File": file,
                "Sentence #": sent_idx,
                "Start Index": start_index,
                "Word": pos_tags[i][0],
                "POS": pos_tags[i][1],
                "Tag": token_tags[token_global_idx],
                "Sentence Text": sentence
            })
            token_global_idx += 1

        sent_idx += 1

df_golden = pd.DataFrame(golden_rows)
df_golden.to_csv(OUTPUT_CSV_GOLDEN, columns=output_columns, index=False)
print(f"Saved Golden CSV to {OUTPUT_CSV_GOLDEN}")


Starting Golden Processing...


Processing Golden Files: 100%|██████████| 50/50 [00:29<00:00,  1.67it/s]


Saved Golden CSV to Golden.csv


## golden validation check

### check B count match

In [26]:
import pandas as pd
import json

# --- 1. Count 'B' tags in the generated Golden.csv ---
df_golden = pd.read_csv('Golden.csv')

# Ensure no hidden whitespace is messing up the count
df_golden['Tag'] = df_golden['Tag'].astype(str).str.strip()

# Count exactly how many 'B' tags exist
csv_b_count = (df_golden['Tag'] == 'B').sum()
print(f"Total 'B' tags in Golden.csv: {csv_b_count}")


# --- 2. Check the theoretical maximum from your JSON ---
golden_labels_file = '/content/annotations_backup (1).json'

with open(golden_labels_file, 'r') as f:
    golden_labels = json.load(f)

json_b_count = 0
for file, labels_data in golden_labels.items():
    # Each item in the 'result' list is one human annotation (one 'B' tag)
    results = labels_data[0].get('result', [])
    json_b_count += len(results)

print(f"Total annotations in JSON:    {json_b_count}")

# --- 3. The Verdict ---
if csv_b_count == json_b_count:
    print("Perfect match! No entities were dropped during conversion.")
else:
    print(f"Mismatch: CSV has {csv_b_count} and JSON has {json_b_count}. Difference: {abs(csv_b_count - json_b_count)}")

Total 'B' tags in Golden.csv: 320
Total annotations in JSON:    320
Perfect match! No entities were dropped during conversion.


In [9]:
# VALIDATION CHECK

import pandas as pd

# Load the newly saved Golden CSV
val_df = pd.read_csv("Golden.csv")

# Define the target file
target_file = "1102.1027v1_Collective Classification of Textual Documents by .tei.txt"

print(f"Validating tags for: {target_file}\n")

# Filter the dataframe for the specific file
file_df = val_df[val_df["File"] == target_file]

# 1. Look for 'BioCreative'
biocreative_rows = file_df[file_df["Word"].str.contains("BioCreative", case=False, na=False)]

Validating tags for: 1102.1027v1_Collective Classification of Textual Documents by .tei.txt



In [10]:
print("--- Checking 'BioCreative' annotations ---")
if not biocreative_rows.empty:
    # Print the word, its tag, and surrounding context
    display(biocreative_rows[["Sentence #", "Word", "Tag", "Sentence Text"]].head(10))
else:
    print("'BioCreative' not found in this file.")

--- Checking 'BioCreative' annotations ---


,Sentence #,Word,Tag,Sentence Text
187,4,BioCreative,B,"More specifically, here we test our model on a..."
430,13,BioCreative,B,"One such effort is the BioCreative challenge, ..."
488,14,BioCreative,O,Machine learning has offered a plethora of sol...
541,16,Biocreative,O,This was the case of the article classificatio...
649,18,Biocreative,B,"Therefore, here we explore the feasibility of ..."
1350,40,BioCreative,O,"Here, we address some of these issues on full-..."
1689,54,Biocreative,O,"In section 5, we describe the biomedical data ..."
4233,134,BioCreative,B,end end end BC 2.5 TRAINING BC 2.5 TESTING Opt...
4258,135,Biocreative,B,The article classification task of Biocreative...
4469,140,Biocreative,B,For final validation we used the entire Biocre...


In [11]:
print("\n--- Summary of ALL captured entities in this file ---")
# Print all words in this file that got a 'B' or 'I' tag
captured_entities = file_df[file_df["Tag"].isin(['B', 'I'])]
if not captured_entities.empty:
    display(captured_entities[["Word", "Tag"]])
else:
    print("No 'B' or 'I' tags were assigned in this file. (Check JSON paths!)")


--- Summary of ALL captured entities in this file ---


,Word,Tag
187,BioCreative,B
430,BioCreative,B
649,Biocreative,B
650,2.5,I
651,.,I
652,data,I
653,set,I
4233,BioCreative,B
4234,(,I
4235,BC,I


# SANITY CHECK

In [12]:
# SANITY CHECK
print("\nRunning Sanity Check...")
rulebased_df = pd.read_csv(OUTPUT_CSV_RULE)
golden_df = pd.read_csv(OUTPUT_CSV_GOLDEN)

assert set(rulebased_df.columns) == set(golden_df.columns), "Columns do not match"

rulebased_df_no_tag = rulebased_df.drop(columns=["Tag"])
golden_df_no_tag = golden_df.drop(columns=["Tag"])

if rulebased_df_no_tag.equals(golden_df_no_tag):
    print("Sanity Check Passed: Both CSVs are aligned correctly!")
else:
    print("Sanity Check Failed: There are discrepancies between the two CSVs.")

    # Using our new composite key for a precise merge check
    merged_df = pd.merge(rulebased_df_no_tag, golden_df_no_tag,
                         on=["File", "Sentence #", "Start Index", "Word", "POS", "Sentence Text"],
                         how="outer", indicator=True)

    mismatched_rows = merged_df[merged_df['_merge'] != 'both']
    print(f"Mismatched rows:\n{mismatched_rows}")


Running Sanity Check...
Sanity Check Passed: Both CSVs are aligned correctly!


In [13]:
# Optional: Download the CSV files (for Google Colab)
files.download(str(OUTPUT_CSV_RULE))
files.download(str(OUTPUT_CSV_GOLDEN))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Eval

In [14]:
!pip install -q seqeval nervaluate scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [15]:
import pandas as pd
from sklearn.metrics import classification_report as sklearn_report
from seqeval.metrics import classification_report as seqeval_report
from nervaluate import Evaluator
import pprint

In [16]:
# Load perfectly aligned data
data = pd.read_csv('Golden.csv')
test = pd.read_csv('RULEBASED_from_txt.csv')

In [17]:
data.head(6)

,File,Sentence #,Start Index,Word,POS,Tag,Sentence Text
0,1102.1027v1_Collective Classification of Textu...,0,0,Collective,JJ,O,Collective Classification of Textual Documents...
1,1102.1027v1_Collective Classification of Textu...,0,0,Classification,NNP,O,Collective Classification of Textual Documents...
2,1102.1027v1_Collective Classification of Textu...,0,0,of,IN,O,Collective Classification of Textual Documents...
3,1102.1027v1_Collective Classification of Textu...,0,0,Textual,NNP,O,Collective Classification of Textual Documents...
4,1102.1027v1_Collective Classification of Textu...,0,0,Documents,NNP,O,Collective Classification of Textual Documents...
5,1102.1027v1_Collective Classification of Textu...,0,0,by,IN,O,Collective Classification of Textual Documents...


In [18]:
test.head(6)

,File,Sentence #,Start Index,Word,POS,Tag,Sentence Text
0,1102.1027v1_Collective Classification of Textu...,0,0,Collective,JJ,O,Collective Classification of Textual Documents...
1,1102.1027v1_Collective Classification of Textu...,0,0,Classification,NNP,O,Collective Classification of Textual Documents...
2,1102.1027v1_Collective Classification of Textu...,0,0,of,IN,O,Collective Classification of Textual Documents...
3,1102.1027v1_Collective Classification of Textu...,0,0,Textual,NNP,O,Collective Classification of Textual Documents...
4,1102.1027v1_Collective Classification of Textu...,0,0,Documents,NNP,O,Collective Classification of Textual Documents...
5,1102.1027v1_Collective Classification of Textu...,0,0,by,IN,O,Collective Classification of Textual Documents...


In [19]:
# Strip any accidental whitespace from the tags to ensure clean matching
data['Tag'] = data['Tag'].astype(str).str.strip()
test['Tag'] = test['Tag'].astype(str).str.strip()

In [20]:
# 1. TOKEN-LEVEL SCORER (Direct Tag Comparison)
print("=== TOKEN-LEVEL CLASSIFICATION (Word by Word) ===")
# This directly compares the tags row-by-row, ignoring sentence boundaries
print(sklearn_report(data['Tag'], test['Tag'], labels=['B', 'I', 'O'], zero_division=0))
print("\n")

=== TOKEN-LEVEL CLASSIFICATION (Word by Word) ===
              precision    recall  f1-score   support

           B       0.49      0.40      0.44       320
           I       0.43      0.52      0.47       527
           O       1.00      1.00      1.00    240393

    accuracy                           1.00    241240
   macro avg       0.64      0.64      0.64    241240
weighted avg       1.00      1.00      1.00    241240





In [21]:
# 2. ENTITY-LEVEL SCORER (Sentence by Sentence)
class SentenceGetter(object):
    def __init__(self, df):
        self.data = df

        # Explicitly define columns to silence the Pandas DeprecationWarning
        cols = ["Word", "POS", "Tag"]
        agg_func = lambda s: [(w, p, t) for w, p, t in zip(s["Word"], s["POS"], s["Tag"])]

        # Apply the function only to our selected columns
        self.grouped = self.data.groupby(["File", "Sentence #"], sort=False)[cols].apply(agg_func)
        self.sentences = self.grouped.tolist()

getter = SentenceGetter(data)
test_getter = SentenceGetter(test)

sentences = getter.sentences
test_sentences = test_getter.sentences

In [22]:
def sent2labels(sent):
    """Formats tags into strict IOB2 format for seqeval (e.g., 'B' -> 'B-DATASET')"""
    formatted_labels = []
    for token, postag, label in sent:
        if label in ["B", "I"]:
            formatted_labels.append(f"{label}-DATASET")
        else:
            formatted_labels.append("O")
    return formatted_labels

In [23]:
y_true = [sent2labels(s) for s in sentences]
y_pred = [sent2labels(s) for s in test_sentences]

In [24]:
print("=== ENTITY-LEVEL CLASSIFICATION (Seqeval) ===")
print(seqeval_report(y_true, y_pred, zero_division=0))
print("\n")

=== ENTITY-LEVEL CLASSIFICATION (Seqeval) ===
              precision    recall  f1-score   support

     DATASET       0.42      0.34      0.38       321

   micro avg       0.42      0.34      0.38       321
   macro avg       0.42      0.34      0.38       321
weighted avg       0.42      0.34      0.38       321





In [25]:
print("=== NERVALUATE SCORES ===")
evaluator = Evaluator(y_true, y_pred, tags=['DATASET'], loader='list')

# Capture the entire output in one variable to avoid unpacking keys
metrics = evaluator.evaluate()

# Check if it returned a tuple (older versions) or a dict (newer versions)
if isinstance(metrics, tuple):
    overall_results = metrics[0]
elif isinstance(metrics, dict):
    overall_results = metrics.get('overall', metrics)
else:
    overall_results = metrics

pprint.pprint(overall_results)

=== NERVALUATE SCORES ===
{'ent_type': EvaluationResult(correct=135,
                              incorrect=0,
                              partial=0,
                              missed=186,
                              spurious=126,
                              precision=0.5172413793103449,
                              recall=0.4205607476635514,
                              f1=0.46391752577319584,
                              actual=261,
                              possible=321),
 'exact': EvaluationResult(correct=110,
                           incorrect=25,
                           partial=0,
                           missed=186,
                           spurious=126,
                           precision=0.421455938697318,
                           recall=0.3426791277258567,
                           f1=0.3780068728522336,
                           actual=261,
                           possible=321),
 'partial': EvaluationResult(correct=110,
                     